# 04b - Pengujian Hipotesis
Materi Modul 4: hipotesis, tingkat signifikansi & nilai-p, kesalahan Tipe 1 vs Tipe 2, statistik F, korelasi vs sebab-akibat.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
peng = pd.read_csv('../dataset/penguins.csv')
dia = pd.read_csv('../dataset/diabetes.csv')
alpha = 0.05

## 1. Uji-t dua sampel
H0: rata-rata `glucose` pasien diabetes = non-diabetes.  H1: berbeda.
(nilai glucose = 0 adalah data tidak valid, jadi dikeluarkan)

In [2]:
d = dia[dia['glucose'] > 0]
diabetes = d.loc[d['target']==1, 'glucose']
sehat = d.loc[d['target']==0, 'glucose']
t, p = stats.ttest_ind(diabetes, sehat, equal_var=False)
print(f'Welch t = {t:.3f}, p-value = {p:.2e}')
print('Keputusan:', 'Tolak H0' if p < alpha else 'Gagal tolak H0')

Welch t = 14.884, p-value = 2.84e-41
Keputusan: Tolak H0


## 2. Alternatif non-parametrik (Mann-Whitney U)

In [3]:
u, p_u = stats.mannwhitneyu(diabetes, sehat)
print(f'U = {u:.1f}, p-value = {p_u:.2e}')

U = 104808.5, p-value = 1.31e-40


## 3. ANOVA & statistik F
H0: rata-rata `body_mass_g` sama di ketiga spesies penguin.

In [4]:
grup = [peng.loc[peng['target']==k, 'body_mass_g'] for k in (0,1,2)]
F, p_f = stats.f_oneway(*grup)
print(f'F = {F:.2f}, p-value = {p_f:.2e}')

F = 341.89, p-value = 3.74e-81


## 4. Uji Chi-square (dua variabel kategorikal)

In [5]:
peng['kelas_massa'] = pd.qcut(peng['body_mass_g'], 3, labels=['ringan','sedang','berat'])
kt = pd.crosstab(peng['kelas_massa'], peng['target'])
chi2, p_c, dof, _ = stats.chi2_contingency(kt)
print(kt); print(f'chi2 = {chi2:.2f}, dof = {dof}, p-value = {p_c:.2e}')

target        0   1    2
kelas_massa             
ringan       77  36    0
sedang       62  31   17
berat         7   1  102
chi2 = 238.41, dof = 4, p-value = 2.04e-50


## 5. Kesalahan Tipe 1 vs Tipe 2 (simulasi)
- Tipe 1: menolak H0 padahal H0 benar (probabilitas = alpha)
- Tipe 2: gagal menolak H0 padahal H1 benar (power = 1 - Tipe 2)

In [6]:
def tolak(mean_b, n=30, sims=2000):
    return np.mean([stats.ttest_ind(rng.normal(0,1,n), rng.normal(mean_b,1,n)).pvalue < alpha for _ in range(sims)])
power = tolak(0.5)
print('H0 benar (selisih 0)    -> laju Tipe 1 :', tolak(0.0))
print('H1 benar (selisih 0.5)  -> power        :', power, '| Tipe 2 =', round(1-power,3))

H0 benar (selisih 0)    -> laju Tipe 1 : 0.0465
H1 benar (selisih 0.5)  -> power        : 0.476 | Tipe 2 = 0.524


## 6. Korelasi vs Sebab-akibat
Contoh klasik: jumlah petugas pemadam dan kerugian kebakaran sama-sama dipengaruhi ukuran kebakaran (confounder).

In [7]:
ukuran = rng.gamma(4, 2, 300)
pemadam = 2*ukuran + rng.normal(0, 2, 300)
kerugian = 5*ukuran + rng.normal(0, 6, 300)
print('Korelasi pemadam vs kerugian:', round(np.corrcoef(pemadam, kerugian)[0,1], 3))
# kontrol ukuran kebakaran: korelasi residual
res_p = pemadam - np.polyval(np.polyfit(ukuran, pemadam, 1), ukuran)
res_k = kerugian - np.polyval(np.polyfit(ukuran, kerugian, 1), ukuran)
print('Setelah ukuran dikontrol    :', round(np.corrcoef(res_p, res_k)[0,1], 3))

Korelasi pemadam vs kerugian: 0.938
Setelah ukuran dikontrol    : 0.036


## Kesimpulan
- (ringkas keputusan tiap uji dan artinya; kenapa korelasi tidak sama dengan kausalitas)